In [3]:
from pathlib import Path
Path("src").mkdir(exist_ok=True)

In [4]:
from pathlib import Path
import os, sys

# Notebooks live in notebooks/, but our code lives in src/.
# Move to the project root so paths like "src/..." work.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

# Auto-reload our modules when we rewrite them (no kernel restarts needed)
%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working directory: C:\Users\makar\Python\auto-data-scientist


In [5]:
%%writefile src/__init__.py
# Marks src/ as a package so we can write: from src.sandbox import ...

Writing src/__init__.py


In [6]:
%%writefile src/config.py
from pathlib import Path

# Project root = the folder that contains src/
ROOT = Path(__file__).resolve().parent.parent

# Every agent run gets its own folder inside runs/
RUNS_DIR = ROOT / "runs"

# Sandbox settings
SANDBOX_IMAGE = "ads-sandbox"        # name of the Docker image we build below
SANDBOX_TIMEOUT_SEC = 90             # kill code that runs longer than this
SANDBOX_MEMORY = "1g"                # RAM limit for the container
SANDBOX_CPUS = "1"                   # CPU limit
MAX_OUTPUT_CHARS = 4000              # cap on output we keep (LLMs have limited context)

Writing src/config.py


In [7]:
%%writefile Dockerfile.sandbox
# Start from a small official Python image
FROM python:3.11-slim

# Install the libraries the AI-written code is allowed to use
RUN pip install --no-cache-dir pandas numpy scikit-learn xgboost matplotlib seaborn joblib

# All code runs in /workspace (we mount a host folder here)
WORKDIR /workspace

Writing Dockerfile.sandbox


In [8]:
# The "!" runs a shell command from inside Jupyter
!docker build -f Dockerfile.sandbox -t ads-sandbox .

^C


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.sandbox
#1 transferring dockerfile: 354B 0.1s done
#1 DONE 0.1s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 12.2s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.1s

#4 [1/3] FROM docker.io/library/python:3.11-slim@sha256:da047cb8f9d1d98e5c070f5300ba9f7274e33b8fc0e5be5ed88740aed1b95ba9
#4 resolve docker.io/library/python:3.11-slim@sha256:da047cb8f9d1d98e5c070f5300ba9f7274e33b8fc0e5be5ed88740aed1b95ba9 0.1s done
#4 DONE 0.2s

#4 [1/3] FROM docker.io/library/python:3.11-slim@sha256:da047cb8f9d1d98e5c070f5300ba9f7274e33b8fc0e5be5ed88740aed1b95ba9
#4 sha256:0526d5e29bf341cb1941d46a6d4ab64fcde4fca3dc3adccfb4dbf5728b88b61e 0B / 249B 0.2s
#4 sha256:58abdd9670ca8c69d03432ce188995dee67f0979357caf83866fe1ef2ec37138 0B / 14.45MB 0.2s
#4 sha256:41e7217c2e506d048f89ca0c5cfef848c05ef990900627c19f99938c4aae57f8 0B / 1.29MB 0.2s

In [9]:
!docker build --progress=plain -f Dockerfile.sandbox -t ads-sandbox .
    

2026/09/24 00:42:01 http2: server: error reading preface from client //./pipe/dockerDesktopLinuxEngine: file has already been closed
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.sandbox
#1 transferring dockerfile: 38B 0.0s
#1 transferring dockerfile: 354B 0.1s done
#1 DONE 0.2s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 2.6s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.1s

#4 [1/3] FROM docker.io/library/python:3.11-slim@sha256:da047cb8f9d1d98e5c070f5300ba9f7274e33b8fc0e5be5ed88740aed1b95ba9
#4 resolve docker.io/library/python:3.11-slim@sha256:da047cb8f9d1d98e5c070f5300ba9f7274e33b8fc0e5be5ed88740aed1b95ba9 0.1s done
#4 sha256:0526d5e29bf341cb1941d46a6d4ab64fcde4fca3dc3adccfb4dbf5728b88b61e 249B / 249B 1.2s done
#4 sha256:41e7217c2e506d048f89ca0c5cfef848c05ef990900627c19f99938c4aae57f8 1.29MB / 1.29MB 7.6s done
#4 sha256:58abdd9670ca8c69d03432ce188995de

In [ ]:
# Testing

In [12]:
from src.sandbox import run_in_sandbox
from src.config import RUNS_DIR

ws = RUNS_DIR / "sandbox_test" / "workspace"

res = run_in_sandbox('import pandas as pd\nprint("Hello from the sandbox! pandas", pd.__version__)', ws)
print(res.summary())
# EXPECTED: ok=True and the hello message

ok=True
returncode=0
STDOUT:
Hello from the sandbox! pandas 3.0.6



In [13]:
res = run_in_sandbox("print(1 / 0)", ws)
print(res.summary())
# EXPECTED: ok=False, STDERR contains "ZeroDivisionError"

ok=False
returncode=1
STDERR:
Traceback (most recent call last):
  File "/workspace/step.py", line 1, in <module>
    print(1 / 0)
          ~~^~~
ZeroDivisionError: division by zero



In [14]:
code = """
import urllib.request
try:
    urllib.request.urlopen("https://example.com", timeout=5)
    print("INTERNET WORKS (bad!)")
except Exception as e:
    print("Blocked as expected:", type(e).__name__)
"""
print(run_in_sandbox(code, ws).summary())


ok=True
returncode=0
STDOUT:
Blocked as expected: URLError



In [15]:
res = run_in_sandbox("while True:\n    pass", ws, timeout=5)
print(res.summary())
#

ok=False
returncode=-1
TIMED OUT
STDERR:
Execution exceeded the 5s time limit and was killed.


In [16]:
res = run_in_sandbox('x = "a" * (3 * 1024**3)\nprint("allocated")', ws)
print(res.summary())


ok=False
returncode=137


In [17]:
code = """
try:
    open("/etc/hacked.txt", "w").write("hi")
    print("WROTE OUTSIDE WORKSPACE (bad!)")
except Exception as e:
    print("Blocked as expected:", type(e).__name__)
"""
print(run_in_sandbox(code, ws).summary())

ok=True
returncode=0
STDOUT:
Blocked as expected: OSError



In [18]:
import pandas as pd, numpy as np

# Put a small fake dataset into the workspace (this is how real data will arrive)
df = pd.DataFrame({"x": np.arange(50), "y": np.arange(50) * 2 + np.random.randn(50)})
df.to_csv(ws / "data.csv", index=False)

code = """
import pandas as pd, matplotlib
matplotlib.use("Agg")          # draw charts without a screen
import matplotlib.pyplot as plt

df = pd.read_csv("data.csv")
print(df.describe())
df.plot(x="x", y="y"); plt.savefig("chart.png")
"""
res = run_in_sandbox(code, ws)
print(res.summary())
# EXPECTED: NEW FILES: ['chart.png']  (and the file exists in runs/sandbox_test/workspace/)

ok=True
returncode=0
STDOUT:
              x          y
count  50.00000  50.000000
mean   24.50000  48.884007
std    14.57738  29.417998
min     0.00000  -0.380308
25%    12.25000  24.228072
50%    24.50000  48.275649
75%    36.75000  72.763213
max    49.00000  97.936168

NEW FILES: ['chart.png']


In [19]:
# tests/test_sandbox.py
from src.sandbox import run_in_sandbox


def test_network_is_blocked(tmp_path):
    """Code inside the sandbox must not be able to reach the internet."""
    # tmp_path is a fresh temporary folder that pytest creates for this test
    # and deletes afterwards. We use it as the sandbox workspace.
    code = """
import urllib.request
try:
    urllib.request.urlopen("https://example.com", timeout=5)
    print("INTERNET WORKS")
except Exception as e:
    print("Blocked:", type(e).__name__)
"""
    result = run_in_sandbox(code, tmp_path)

    # 1) The dangerous outcome must not happen
    assert "INTERNET WORKS" not in result.stdout

    # 2) Also check that the block really happened. Without this line the test
    #    would pass even if the sandbox crashed before running our code at all.
    assert "Blocked" in result.stdout

In [20]:
def test_errors_are_captured(tmp_path):
    result = run_in_sandbox("print(1 / 0)", tmp_path)
    assert not result.ok
    assert "ZeroDivisionError" in result.stderr


def test_infinite_loop_is_killed(tmp_path):
    result = run_in_sandbox("while True:\n    pass", tmp_path, timeout=5)
    assert result.timed_out
    assert not result.ok

In [22]:
from pathlib import Path
print(Path("pytest.ini").exists(), "->", Path("pytest.ini").resolve())

True -> C:\Users\makar\Python\auto-data-scientist\pytest.ini


In [23]:
from pathlib import Path

print("tests folder exists:", Path("tests").exists())
print("files in tests/    :", [p.name for p in Path("tests").glob("*")])
print("stray copies       :", [str(p) for p in Path(".").rglob("test_sandbox*")])

tests folder exists: False
files in tests/    : []
stray copies       : []


In [24]:
from pathlib import Path
import os

# Make sure we're in the project root (same guard as Cell 1)
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

Path("tests").mkdir(exist_ok=True)   # creates the folder if it's missing
print("Working directory:", Path.cwd())
print("tests exists:", Path("tests").exists())

Working directory: C:\Users\makar\Python\auto-data-scientist
tests exists: True


In [25]:
%%writefile tests/test_sandbox.py
from src.sandbox import run_in_sandbox


def test_network_is_blocked(tmp_path):
    """Code inside the sandbox must not be able to reach the internet."""
    code = """
import urllib.request
try:
    urllib.request.urlopen("https://example.com", timeout=5)
    print("INTERNET WORKS")
except Exception as e:
    print("Blocked:", type(e).__name__)
"""
    result = run_in_sandbox(code, tmp_path)

    # The dangerous outcome must not happen
    assert "INTERNET WORKS" not in result.stdout
    # And the block must really have happened (guards against a silent crash)
    assert "Blocked" in result.stdout

Writing tests/test_sandbox.py


In [26]:
from src.sandbox import run_in_sandbox
from src.config import RUNS_DIR

ws = RUNS_DIR / "sandbox_test" / "workspace"

# name -> (code as text, timeout in seconds)
tests = {
    "8  error captured": ("print(1 / 0)", 60),
    "10 infinite loop":  ("while True:\n    pass", 5),
    "11 memory bomb":    ('x = "a" * (3 * 1024**3)\nprint("allocated")', 60),
    "12 read-only fs":   ('''
try:
    open("/etc/hacked.txt", "w").write("hi")
    print("WROTE OUTSIDE WORKSPACE (bad!)")
except Exception as e:
    print("Blocked:", type(e).__name__)
''', 60),
}

for name, (code, timeout) in tests.items():
    r = run_in_sandbox(code, ws, timeout=timeout)
    print(f"{name:20} ok={r.ok}  returncode={r.returncode}  timed_out={r.timed_out}")
    print("    stdout:", r.stdout.strip()[:100])
    last_err = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ""
    print("    stderr:", last_err)

8  error captured    ok=False  returncode=1  timed_out=False
    stdout: 
    stderr: ZeroDivisionError: division by zero
10 infinite loop     ok=False  returncode=-1  timed_out=True
    stdout: 
    stderr: Execution exceeded the 5s time limit and was killed.
11 memory bomb       ok=False  returncode=137  timed_out=False
    stdout: 
    stderr: 
12 read-only fs      ok=True  returncode=0  timed_out=False
    stdout: Blocked: OSError
    stderr: 
